# 第一阶段 S1-D12A：BC 与 PPO 训练代码

这份笔记本按“场景 → 观测 → 选动作 → 环境给奖励 → 教师/BC → PPO 更新 → 工作台训练循环”的顺序，直接展示 **2026-10-10 正式实验所用的原始 Python 定义**。代码来自 Git 提交 `dfe7c394267e94bd2084ae72add3aa2c855b578f`，与 [冻结 run_plan](../../../outputs/experiments/first_stage_s1_d12a_20261010/run_plan.json) 和 [完成 run_manifest](../../../outputs/experiments/first_stage_s1_d12a_20261010/run_manifest.json) 对应。文中“运行代码”只指把类和函数装入内存；**Run All 不会调用训练、评价、工作台 API，也不会加载旧权重重新跑实验**。

读者可先看每段前面的中文说明，再展开对应源码。`%%project_module` 会核对代码单元与冻结 Git 文件的逐行内容，把定义放进原来的模块命名空间。这样跨文件的相对导入仍按原项目工作，也能防止展示代码悄悄偏离正式版本。运行需要本地 Git 能读取该提交，并需要 DQN 环境或其他具备项目依赖、IPython、PyTorch 的 Python 内核。


## 0. 装载冻结源码，只做来源核验

下面的初始化单元把提交中的 `src/satellite_rl/` 导出到 `outputs/training_notebooks_20261012/stage1/`。之后所有 `satellite_rl` 导入都指向这个隔离副本。若内核已从别的路径导入同名包，单元会报错，提示重启内核，以免混入当前工作树代码。没有网络请求，也不启动任务。


In [1]:
from pathlib import Path
import hashlib
import importlib
import io
import json
import subprocess
import sys
import types
import zipfile
from IPython.core.magic import register_cell_magic

SCIENTIFIC_COMMIT = "dfe7c394267e94bd2084ae72add3aa2c855b578f"
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / ".git").exists() and (p / "EXPERIMENT_STATUS.md").exists()), None)
if REPO is None:
    raise FileNotFoundError("请从 D:/paper 或其子目录启动笔记本")
RUN_PLAN = REPO / "outputs/experiments/first_stage_s1_d12a_20261010/run_plan.json"
RUN_MANIFEST = REPO / "outputs/experiments/first_stage_s1_d12a_20261010/run_manifest.json"
plan = json.loads(RUN_PLAN.read_text(encoding="utf-8"))
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))
assert plan["formal_source_commit"] == manifest["scientific_commit"] == SCIENTIFIC_COMMIT
resolved = subprocess.check_output(
    ["git", "rev-parse", SCIENTIFIC_COMMIT + "^{commit}"], cwd=REPO,
    text=True, encoding="utf-8").strip()
assert resolved == SCIENTIFIC_COMMIT

FROZEN = REPO / "outputs/training_notebooks_20261012/stage1" / ("frozen_" + SCIENTIFIC_COMMIT)
FROZEN.mkdir(parents=True, exist_ok=True)
archive = subprocess.check_output(
    ["git", "archive", "--format=zip", SCIENTIFIC_COMMIT, "src/satellite_rl"], cwd=REPO)
with zipfile.ZipFile(io.BytesIO(archive)) as zf:
    for entry in zf.infolist():
        path = Path(entry.filename)
        if path.parts == ("src",) and entry.is_dir():
            continue
        if path.parts[:2] != ("src", "satellite_rl") or ".." in path.parts:
            raise ValueError("冻结归档含意外路径")
        target = FROZEN / path
        if entry.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(entry))
FROZEN_SRC = FROZEN / "src"
for name, module in tuple(sys.modules.items()):
    if name == "satellite_rl" or name.startswith("satellite_rl."):
        module_file = getattr(module, "__file__", None)
        if module_file and not Path(module_file).resolve().is_relative_to(FROZEN_SRC.resolve()):
            raise RuntimeError("内核已加载别处的 satellite_rl；请重启内核后 Run All")
if str(FROZEN_SRC) not in sys.path:
    sys.path.insert(0, str(FROZEN_SRC))

SOURCE_RANGES = {}
@register_cell_magic
def project_module(line, cell):
    name, span = line.split()
    if not name.startswith("satellite_rl.coverage.drl_r04_"):
        raise ValueError("只允许加载本笔记本的 D12-A 原模块")
    lo, hi = (int(x) for x in span.split(":"))
    path = FROZEN_SRC / Path(*name.split(".")).with_suffix(".py")
    lines = path.read_text(encoding="utf-8").splitlines(keepends=True)
    expected = "".join(lines[lo - 1:hi])
    if cell.rstrip("\n") != expected.rstrip("\n"):
        raise ValueError(f"源码单元与冻结提交不一致：{name}:{lo}-{hi}")
    if name not in sys.modules:
        importlib.import_module(name.rpartition(".")[0])
        module = types.ModuleType(name)
        module.__file__ = str(path)
        module.__package__ = name.rpartition(".")[0]
        sys.modules[name] = module
    module = sys.modules[name]
    if Path(module.__file__).resolve() != path.resolve():
        raise RuntimeError(f"模块来源不符：{name}")
    exec(compile(cell, str(path), "exec"), module.__dict__)
    SOURCE_RANGES.setdefault(name, []).append((lo, hi))
    print(f"已核冻结源码并装载定义：{name.rsplit('.', 1)[-1]} 第 {lo}–{hi} 行")

print("冻结提交：", SCIENTIFIC_COMMIT)
print("隔离源码：", FROZEN_SRC)
print("实验状态：", manifest["status"])


冻结提交： dfe7c394267e94bd2084ae72add3aa2c855b578f
隔离源码： D:\paper\outputs\training_notebooks_20261012\stage1\frozen_dfe7c394267e94bd2084ae72add3aa2c855b578f\src
实验状态： delivered


## 1. 固定的问题与环境预算

本实验处理静态、固定 0.5° 波束宽度的布局改善。每个场景从 R04 初始布局出发，最多 200 束；策略只在 `STOP`、`split`、`merge` 候选中选一个，最多提交 32 次动作、试探 256 个候选、记账 8192 次几何调用。`DrlR04Config` 把这些限制写成不可随意改动的约束。它不是最终完整课题：波束宽度学习与动态业务尚未在此组实现。


In [2]:
%%project_module satellite_rl.coverage.drl_r04_config 1:35
"""Frozen A-stage contract for R04-guided layout improvement."""

from dataclasses import dataclass

from .variable_config import VariableConfig


@dataclass(frozen=True)
class DrlR04Config:
    max_beams: int = 200
    width_deg: float = 0.5
    max_commits: int = 32
    max_trials: int = 256
    max_geometry_calls: int = 8192
    max_candidates: int = 32
    gamma: float = 1.0
    reward_scale: float = 100.0
    menu_version: str = "r04_split_merge_local_v1"
    objective_version: str = "r04_reference_pressure_v1"

    def __post_init__(self):
        if not 1 <= self.max_beams <= 200:
            raise ValueError("max_beams must be in 1..200")
        for name in ("max_commits", "max_trials", "max_geometry_calls", "max_candidates"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name} must be positive")
        if self.width_deg != 0.5 or self.gamma != 1.0 or self.reward_scale != 100.0:
            raise ValueError("A-stage width, discount and reward scale are frozen")
        if self.menu_version != "r04_split_merge_local_v1" or self.objective_version != "r04_reference_pressure_v1":
            raise ValueError("unknown A-stage menu or objective")

    def variable_config(self) -> VariableConfig:
        return VariableConfig(max_beams=self.max_beams, fixed_width_deg=self.width_deg,
                              max_actions=self.max_commits, max_candidates=self.max_candidates,
                              max_geometry_calls=self.max_geometry_calls)


已核冻结源码并装载定义：drl_r04_config 第 1–35 行


## 2. 把布局变成策略能看的数字

`encode` 不偷偷执行候选动作，也不给网络候选的真实目标值。它把现有波束写成 `N × 12` 特征、全局状态写成 14 维、当前动作菜单写成 `K × 16` 特征；`N` 是当前波束数，`K` 是候选动作数。负载、束中心、需求、当前与最好目标值、剩余预算都能进入观察。`candidate_mask` 表示可选择的菜单项。候选的收益要等环境真正 `step` 后才知道。


In [3]:
%%project_module satellite_rl.coverage.drl_r04_observation 1:72
"""Cheap pre-action features for the fixed-width R04 improvement task."""

import hashlib
import json

import numpy as np

from .variable_design import Layout


BEAM_FEATURES = 12
CANDIDATE_FEATURES = 16
GLOBAL_FEATURES = 14


def layout_sha256(layout: Layout) -> str:
    payload = json.dumps(layout.to_record(), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def candidate_sha256(menu) -> str:
    payload = json.dumps([action.to_record() for action, _ in menu],
                         sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def encode(scene, layout, cfg, contract, *, current_j, best_j, best_n,
           commits, trials, geometry_calls, menu):
    """No candidate execution, objective lookup or exact ray matrix occurs here."""
    positions = np.asarray(scene.positions, dtype=np.float64)
    demands = np.asarray(scene.demands, dtype=np.float64)
    owner = np.asarray(layout.assignment, dtype=np.int64)
    n = len(layout.centers)
    valid = owner >= 0
    loads = np.bincount(owner[valid], weights=demands[valid], minlength=n)
    counts = np.bincount(owner[valid], minlength=n)
    centroid = np.zeros((n, 2), dtype=np.float64)
    if n:
        np.add.at(centroid, owner[valid], positions[valid] * demands[valid, None])
        centroid /= np.maximum(loads[:, None], 1e-9)
    beam = np.zeros((n, BEAM_FEATURES), dtype=np.float32)
    if n:
        beam[:, 0:2] = np.asarray(layout.centers, dtype=np.float32) / (90.0, 180.0)
        beam[:, 2] = np.asarray(layout.widths) / cfg.fixed_width_deg
        beam[:, 3] = loads / cfg.reference_capacity_bps
        beam[:, 4] = np.log1p(loads / cfg.reference_capacity_bps)
        beam[:, 5] = counts / max(len(demands), 1)
        beam[:, 6:8] = (centroid - layout.centers) / (10.0, 10.0)
        beam[:, 8] = (loads > cfg.reference_capacity_bps).astype(np.float32)
        beam[:, 9] = loads / max(float(demands.sum()), 1e-9)
        beam[:, 10] = np.arange(n) / max(n, 1)
        beam[:, 11] = 1.0
    ratio = loads / cfg.reference_capacity_bps
    cv = float(loads.std() / loads.mean()) if n and loads.mean() > 0 else 0.0
    global_features = np.asarray([
        n / cfg.max_beams, float(valid.mean()) if len(valid) else 1.0,
        float(demands[valid].sum() / demands.sum()) if demands.sum() else 1.0,
        cv, float(ratio.max()) if n else 0.0,
        commits / contract.max_commits, geometry_calls / contract.max_geometry_calls,
        np.log1p(float(demands.sum()) / cfg.reference_capacity_bps),
        float(current_j), float(best_j),
        (contract.max_commits - commits) / contract.max_commits,
        (contract.max_trials - trials) / contract.max_trials,
        (contract.max_geometry_calls - geometry_calls) / contract.max_geometry_calls,
        best_n / cfg.max_beams,
    ], dtype=np.float32)
    features = np.stack([np.asarray(f, dtype=np.float32) for _, f in menu]) if menu else np.zeros((1, CANDIDATE_FEATURES), dtype=np.float32)
    mask = np.ones(len(menu), dtype=bool) if menu else np.ones(1, dtype=bool)
    if not np.isfinite(beam).all() or not np.isfinite(features).all() or not np.isfinite(global_features).all():
        raise ValueError("nonfinite R04 observation")
    return {"beam": beam, "candidate": features, "candidate_mask": mask,
            "global": global_features}


已核冻结源码并装载定义：drl_r04_observation 第 1–72 行


## 3. 网络怎样选择动作；PPO 更新哪个损失

`collate` 只在一个小批内补齐不同的束数和动作数；掩码排除补齐项。`DrlR04Policy` 先编码每束，再汇总成不依赖束顺序的上下文，给每个候选打分。`act` 在训练时按分类分布采样动作，在验证时可取最高分；价值头估计以后还能得到多少回报。此模型正式终点有 88450 个参数。

`Transition` 保存一次实际交互的观察、动作、旧对数概率、价值估计、奖励和结束标记。`returns_and_advantages` 沿各自环境流倒推 GAE；`ppo_update` 计算裁剪后的策略目标、价值误差与熵奖励，做 4 轮小批梯度更新。这里 PPO 学的是**选择布局操作**，不是直接预测束中心或功率。


In [4]:
%%project_module satellite_rl.coverage.drl_r04_policy 1:149
"""D12-A set policy and PPO updates; no environment or runner dependency.

Scene geometry and candidate construction belong to variable_observation.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical



BEAM_FEATURES = 12
CANDIDATE_FEATURES = 16
GLOBAL_FEATURES = 14


def collate(observations, device):
    """Pad only for a batch; padding is excluded from set pooling and logits."""
    if not observations:
        raise ValueError("Cannot collate an empty PPO batch")
    b = len(observations)
    max_beams = max(1, max(len(item["beam"]) for item in observations))
    max_candidates = max(len(item["candidate"]) for item in observations)
    beams = np.zeros((b, max_beams, BEAM_FEATURES), dtype=np.float32)
    beam_mask = np.zeros((b, max_beams), dtype=bool)
    actions = np.zeros((b, max_candidates, CANDIDATE_FEATURES), dtype=np.float32)
    action_mask = np.zeros((b, max_candidates), dtype=bool)
    for i, item in enumerate(observations):
        n, k = len(item["beam"]), len(item["candidate"])
        beams[i, :n] = item["beam"]
        beam_mask[i, :n] = True
        actions[i, :k] = item["candidate"]
        action_mask[i, :k] = item["candidate_mask"]
    return {"beam": torch.as_tensor(beams, device=device),
            "beam_mask": torch.as_tensor(beam_mask, device=device),
            "global": torch.as_tensor(np.stack([x["global"] for x in observations]), device=device),
            "candidate": torch.as_tensor(actions, device=device),
            "candidate_mask": torch.as_tensor(action_mask, device=device)}


class DrlR04Policy(nn.Module):
    """Shared beam encoder, permutation-invariant context and candidate scorer."""

    def __init__(self, hidden_size=128):
        super().__init__()
        if hidden_size != 128:
            raise ValueError("D12-A policy uses hidden_size=128")
        self.beam_encoder = nn.Sequential(nn.Linear(BEAM_FEATURES, 128), nn.Tanh(),
                                          nn.Linear(128, 128), nn.Tanh())
        self.context = nn.Sequential(nn.Linear(128 + GLOBAL_FEATURES, 128), nn.Tanh())
        self.candidate_encoder = nn.Sequential(nn.Linear(CANDIDATE_FEATURES, 128), nn.Tanh(),
                                               nn.Linear(128, 128), nn.Tanh())
        self.query = nn.Linear(128, 128)
        self.score = nn.Linear(128, 1)
        self.value_head = nn.Sequential(nn.Linear(128, 128), nn.Tanh(), nn.Linear(128, 1))

    def forward(self, batch):
        beam = self.beam_encoder(batch["beam"])
        mask = batch["beam_mask"].unsqueeze(-1)
        pooled = (beam * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        context = self.context(torch.cat((pooled, batch["global"]), dim=-1))
        embedded = self.candidate_encoder(batch["candidate"])
        logits = self.score(torch.tanh(embedded + self.query(context)[:, None])).squeeze(-1)
        logits = logits.masked_fill(~batch["candidate_mask"], torch.finfo(logits.dtype).min)
        return logits, self.value_head(context).squeeze(-1)

    def act(self, batch, *, deterministic=False):
        logits, value = self(batch)
        distribution = Categorical(logits=logits)
        choice = logits.argmax(dim=-1) if deterministic else distribution.sample()
        return choice, distribution.log_prob(choice), value

    def evaluate(self, batch, choice):
        logits, value = self(batch)
        distribution = Categorical(logits=logits)
        return distribution.log_prob(choice), distribution.entropy(), value


@dataclass
class Transition:
    observation: dict
    choice: int
    old_log_prob: float
    value: float
    reward: float
    done: bool


def returns_and_advantages(transitions, bootstrap, *, num_envs, gamma, gae_lambda):
    """GAE over time×environment; neighboring streams never bootstrap each other."""
    if not transitions:
        raise ValueError("No real interactions to update from")
    if len(transitions) % num_envs:
        raise ValueError("Rollout interaction count is not divisible by environment count")
    horizon = len(transitions) // num_envs
    values = np.asarray([x.value for x in transitions], dtype=np.float64).reshape(horizon, num_envs)
    rewards = np.asarray([x.reward for x in transitions], dtype=np.float64).reshape(horizon, num_envs)
    dones = np.asarray([x.done for x in transitions], dtype=np.float64).reshape(horizon, num_envs)
    bootstrap = np.broadcast_to(np.asarray(bootstrap, dtype=np.float64), (num_envs,))
    next_values = np.vstack((values[1:], bootstrap[None, :]))
    advantages = np.zeros((horizon, num_envs), dtype=np.float64)
    gae = np.zeros(num_envs, dtype=np.float64)
    for i in range(horizon - 1, -1, -1):
        continuation = 1.0 - dones[i]
        delta = rewards[i] + gamma * next_values[i] * continuation - values[i]
        gae = delta + gamma * gae_lambda * continuation * gae
        advantages[i] = gae
    return (advantages + values).reshape(-1), advantages.reshape(-1)


def ppo_update(policy, optimizer, transitions, bootstrap, *, device,
               gamma=.99, gae_lambda=.95, clip_ratio=.2, entropy_coef=.01,
               value_coef=.5, max_grad_norm=.5, epochs=4, minibatch_size=64,
               num_envs=1):
    return_values, advantage_values = returns_and_advantages(
        transitions, bootstrap, num_envs=num_envs,
        gamma=gamma, gae_lambda=gae_lambda)
    returns = torch.as_tensor(return_values, dtype=torch.float32, device=device)
    advantage = torch.as_tensor(advantage_values, dtype=torch.float32, device=device)
    advantage = (advantage - advantage.mean()) / advantage.std(unbiased=False).clamp_min(1e-8)
    old_logp = torch.as_tensor([x.old_log_prob for x in transitions], dtype=torch.float32, device=device)
    choices = torch.as_tensor([x.choice for x in transitions], dtype=torch.int64, device=device)
    summaries = []
    for _ in range(epochs):
        for indices in torch.randperm(len(transitions), device=device).split(minibatch_size):
            selected = indices.cpu().tolist()
            batch = collate([transitions[i].observation for i in selected], device)
            logp, entropy, value = policy.evaluate(batch, choices[indices])
            ratio = torch.exp(logp - old_logp[indices])
            objective = torch.minimum(ratio * advantage[indices],
                                      ratio.clamp(1 - clip_ratio, 1 + clip_ratio) * advantage[indices])
            policy_loss = -objective.mean()
            value_loss = .5 * (value - returns[indices]).square().mean()
            loss = policy_loss + value_coef * value_loss - entropy_coef * entropy.mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            norm = nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
            optimizer.step()
            summaries.append((float(policy_loss.detach()), float(value_loss.detach()),
                              float(entropy.mean().detach()), float(norm.detach())))
    mean = np.mean(summaries, axis=0)
    return {"policy_loss": float(mean[0]), "value_loss": float(mean[1]),
            "entropy": float(mean[2]), "grad_norm": float(mean[3]),
            "minibatch_updates": len(summaries), "rollout_interactions": len(transitions)}


已核冻结源码并装载定义：drl_r04_policy 第 1–149 行


## 4. 环境怎样判断动作好坏

`r04_objective` 对完整分配计算 `J = 超容量压力 + 0.1 × 负载比例方差 + 0.02 × 束数/200`，越小越好。`DrlR04Env.reset` 建立共同 R04 初解并检查全部飞机归属；`observe` 生成菜单和上面的观测；`preview` 在预算内真实试探拆束/合束；`step` 执行选中的菜单项，更新当前布局与历来最好布局。奖励是 `100 × (旧的最好 J − 新的最好 J)`，没有改善就为零。STOP、动作数或几何预算会结束一景。教师和 H1 使用同一个执行器，只是动作选择规则不同。


In [5]:
%%project_module satellite_rl.coverage.drl_r04_core 1:50
"""Shared finite-budget R04 executor for learning and rule baselines.

Preview is an explicitly charged operation for offline teachers and search.
The learning policy sees only ``observe``; it never receives preview scores.
"""

from __future__ import annotations

from dataclasses import dataclass
import hashlib
import json
from time import perf_counter

import numpy as np

from .drl_r04_config import DrlR04Config
from .drl_r04_observation import candidate_sha256, encode, layout_sha256
from .variable_design import (Layout, VariableAction, apply_action, candidates,
                              geometry_audit, initial_layout)


# R04 initialization uses three exact matrices. Local menu uses one. The
# split/merge executor currently uses three; reserve eight before each trial
# so an upstream implementation change cannot consume the final few calls.
INITIAL_MATRIX_BOUND = 3
MENU_MATRIX_BOUND = 1
TRIAL_MATRIX_RESERVE = 8


def r04_objective(layout: Layout, demands, reference_capacity_bps: float,
                  max_beams: int) -> float:
    """Exact R04 objective for certified complete layouts, without a new ray call."""
    owner = np.asarray(layout.assignment)
    demand = np.asarray(demands, dtype=np.float64)
    if len(owner) != len(demand) or np.any(owner < 0):
        raise ValueError("R04 objective requires complete assignment")
    loads = np.bincount(owner, weights=demand, minlength=len(layout.centers))
    ratio = loads / reference_capacity_bps
    pressure = float(np.mean(np.maximum(ratio - 1, 0) ** 2)) if len(ratio) else 0.0
    balance = float(np.var(ratio)) if len(ratio) else 0.0
    return pressure + 0.1 * balance + 0.02 * len(ratio) / max_beams


@dataclass
class Trial:
    layout: Layout
    score: float | None
    info: dict




已核冻结源码并装载定义：drl_r04_core 第 1–50 行


In [6]:
%%project_module satellite_rl.coverage.drl_r04_core 51:317
class DrlR04Env:
    """One scene, shared candidate and geometry ledger, and best-so-far reward."""

    def __init__(self, scene, config: DrlR04Config | None = None):
        self.scene = scene
        self.config = config or DrlR04Config()
        self.cfg = self.config.variable_config()
        self.initial_layout = None
        self.current_layout = None
        self.best_layout = None
        self.initial_sha256 = None
        self.scene_sha256 = None
        self.initial_j = None
        self.current_j = None
        self.best_j = None
        self.failure_reason = None
        self.stop_reason = None
        self.done = False
        self.terminated = False
        self.truncated = False
        self.commits = 0
        self.trials = 0
        self.preview_requests = 0
        self.preview_cache_hits = 0
        self.preview_budget_rejections = 0
        self.geometry_calls = 0
        self.matrix_calls = 0
        self.ray_pair_checks = 0
        self.ledger = []
        self.initialization_seconds = 0.0
        self.menu_seconds = 0.0
        self.trial_seconds = 0.0
        self._menu_cache = {}
        self._trial_cache = {}

    def reset(self):
        self.commits = self.trials = self.geometry_calls = 0
        self.preview_requests = self.preview_cache_hits = self.preview_budget_rejections = 0
        self.matrix_calls = self.ray_pair_checks = 0
        self.ledger = []
        self._menu_cache = {}
        self._trial_cache = {}
        self.failure_reason = None
        self.stop_reason = None
        self.done = self.terminated = self.truncated = False
        payload = {"positions": np.asarray(self.scene.positions, dtype=np.float64).tolist(),
                   "demands": np.asarray(self.scene.demands, dtype=np.float64).tolist(),
                   "aircraft_ids": list(getattr(self.scene, "aircraft_ids", ())) }
        self.scene_sha256 = hashlib.sha256(json.dumps(payload, sort_keys=True,
                                                       separators=(",", ":")).encode()).hexdigest()
        started = perf_counter()
        with geometry_audit() as audit:
            if self.config.max_geometry_calls < INITIAL_MATRIX_BOUND:
                layout = Layout.empty(len(self.scene.demands))
                layout.aircraft_ids = tuple(getattr(self.scene, "aircraft_ids", ()))
                init = {"status": "initialization_budget_exhausted",
                        "failure_reason": "initialization_geometry_budget_below_reserved_bound",
                        "metrics": {"coverage_100": False, "demand_conserved": False,
                                    "aircraft_count": len(self.scene.demands),
                                    "total_demand_bps": float(np.sum(self.scene.demands))}}
            else:
                try:
                    layout, init = initial_layout(self.scene, self.cfg, "R04")
                except ValueError as exc:
                    # An overlarge candidate set or impossible constructor remains
                    # a counted scene, with its original aircraft denominator.
                    layout = Layout.empty(len(self.scene.demands))
                    layout.aircraft_ids = tuple(getattr(self.scene, "aircraft_ids", ()))
                    init = {"status": "initialization_exception", "failure_reason": str(exc),
                            "metrics": {"coverage_100": False, "demand_conserved": False,
                                        "aircraft_count": len(self.scene.demands),
                                        "total_demand_bps": float(np.sum(self.scene.demands))}}
        self.initialization_seconds = perf_counter() - started
        self.menu_seconds = self.trial_seconds = 0.0
        self.matrix_calls += audit["matrix_calls"]
        self.ray_pair_checks += audit["ray_pair_checks"]
        self.geometry_calls += max(int(init.get("geometry_budget_charges", 0)),
                                   audit["matrix_calls"])
        if audit["matrix_calls"] > INITIAL_MATRIX_BOUND:
            raise RuntimeError("R04 initialization exceeded reserved matrix bound")
        self.initial_layout = self.current_layout = self.best_layout = layout
        self.initial_sha256 = layout_sha256(layout)
        self.initialization = init
        if not init["metrics"]["coverage_100"] or not init["metrics"]["demand_conserved"]:
            self.failure_reason = init.get("failure_reason") or "initial_layout_incomplete"
            self.stop_reason = "initialization_failed"
            self.done = self.terminated = True
            self.initial_j = self.current_j = self.best_j = None
            return self._failure_observation(), self._info("initialization_failed")
        if len(layout.centers) > self.config.max_beams or not np.allclose(layout.widths, 0.5):
            raise ValueError("initial layout violates A-stage fixed-width cap")
        self.initial_j = self.current_j = self.best_j = r04_objective(
            layout, self.scene.demands, self.cfg.reference_capacity_bps, self.cfg.max_beams)
        if self.geometry_calls + MENU_MATRIX_BOUND + TRIAL_MATRIX_RESERVE > self.config.max_geometry_calls:
            self.stop_reason = "initial_geometry_budget_exhausted"
            self.done = self.truncated = True
            return self._failure_observation(), self._info("initial_geometry_budget_exhausted")
        return self.observe()[0], self._info("ready")

    def _failure_observation(self):
        return {"beam": np.zeros((0, 12), dtype=np.float32),
                "candidate": np.zeros((1, 16), dtype=np.float32),
                "candidate_mask": np.ones(1, dtype=bool),
                "global": np.zeros(14, dtype=np.float32)}

    def _info(self, status, **extra):
        return {"status": status, "failure_reason": self.failure_reason,
                "stop_reason": self.stop_reason,
                "scene_id": getattr(self.scene, "scene_id", None),
                "scene_source_sha256": getattr(self.scene, "source_hash", None),
                "scene_sha256": self.scene_sha256,
                "initial_sha256": self.initial_sha256,
                "current_sha256": layout_sha256(self.current_layout) if self.current_layout is not None else None,
                "best_sha256": layout_sha256(self.best_layout) if self.best_layout is not None else None,
                "initial_j": self.initial_j, "current_j": self.current_j, "best_j": self.best_j,
                "commits": self.commits, "trials": self.trials,
                "preview_requests": self.preview_requests,
                "preview_cache_hits": self.preview_cache_hits,
                "preview_budget_rejections": self.preview_budget_rejections,
                "geometry_calls": self.geometry_calls, "matrix_calls": self.matrix_calls,
                "ray_pair_checks": self.ray_pair_checks,
                "initialization_seconds": self.initialization_seconds,
                "menu_seconds": self.menu_seconds,
                "trial_seconds": self.trial_seconds, **extra}

    def _charge(self, audit, charge):
        self.matrix_calls += audit["matrix_calls"]
        self.ray_pair_checks += audit["ray_pair_checks"]
        self.geometry_calls += max(int(charge), audit["matrix_calls"])

    def _menu(self, layout):
        key = layout_sha256(layout)
        if key in self._menu_cache:
            return self._menu_cache[key]
        if self.geometry_calls + MENU_MATRIX_BOUND + TRIAL_MATRIX_RESERVE > self.config.max_geometry_calls:
            menu = [(VariableAction("stop"), np.zeros(16, dtype=np.float32))]
        else:
            started = perf_counter()
            with geometry_audit() as audit:
                raw = candidates(self.scene, layout, self.cfg, mode="local")
            self.menu_seconds += perf_counter() - started
            if audit["matrix_calls"] > MENU_MATRIX_BOUND:
                raise RuntimeError("R04 menu exceeded reserved matrix bound")
            self._charge(audit, 1)
            menu = [(a, f) for a, f in raw if a.operation in ("stop", "split", "merge")]
        if not menu or menu[0][0].operation != "stop":
            raise ValueError("R04 candidate menu lacks STOP")
        self._menu_cache[key] = menu
        return menu

    @property
    def menu(self):
        return self._menu(self.current_layout) if not self.done else []

    def observe(self):
        if self.current_layout is None:
            raise RuntimeError("reset environment before observe")
        if self.done:
            return self._failure_observation(), []
        menu = self._menu(self.current_layout)
        obs = encode(self.scene, self.current_layout, self.cfg, self.config,
                     current_j=self.current_j, best_j=self.best_j,
                     best_n=len(self.best_layout.centers), commits=self.commits,
                     trials=self.trials, geometry_calls=self.geometry_calls, menu=menu)
        return obs, menu

    def preview(self, action: VariableAction, *, layout: Layout | None = None) -> Trial:
        """Charged real trial, cached for a later commit; never expose to policy input."""
        if self.done:
            raise RuntimeError("episode ended")
        base = layout if layout is not None else self.current_layout
        menu = self._menu(base)
        if action.operation == "stop" or action not in [a for a, _ in menu]:
            raise ValueError("preview requires a menu split/merge action")
        key = (layout_sha256(base), action)
        self.preview_requests += 1
        if key in self._trial_cache:
            self.preview_cache_hits += 1
            return self._trial_cache[key]
        if self.trials >= self.config.max_trials:
            self.preview_budget_rejections += 1
            raise RuntimeError("candidate trial budget exhausted")
        # Existing executor makes at most a few ray matrices per split/merge.
        # Leave headroom so observed calls cannot cross the hard ceiling.
        if self.geometry_calls + TRIAL_MATRIX_RESERVE > self.config.max_geometry_calls:
            self.preview_budget_rejections += 1
            raise RuntimeError("geometry budget exhausted")
        self.trials += 1
        started = perf_counter()
        with geometry_audit() as audit:
            proposed, raw = apply_action(self.scene, base, self.cfg, action)
        self.trial_seconds += perf_counter() - started
        if audit["matrix_calls"] > TRIAL_MATRIX_RESERVE:
            raise RuntimeError("R04 trial exceeded reserved matrix bound")
        self._charge(audit, raw.get("geometry_budget_charges", 0))
        accepted = bool(raw["accepted"])
        score = (r04_objective(proposed, self.scene.demands,
                               self.cfg.reference_capacity_bps, self.cfg.max_beams)
                 if accepted else None)
        info = {**raw, "trial_number": self.trials, "candidate_id": key[0] + ":" +
                hashlib.sha256(json.dumps(action.to_record(), sort_keys=True,
                                          separators=(",", ":")).encode()).hexdigest(),
                "base_sha256": key[0], "trial_sha256": layout_sha256(proposed),
                "trial_j": score, "matrix_calls": audit["matrix_calls"],
                "ray_pair_checks": audit["ray_pair_checks"]}
        trial = Trial(proposed, score, info)
        self._trial_cache[key] = trial
        return trial

    def step(self, action_index: int):
        if self.done:
            raise RuntimeError("episode ended")
        obs, menu = self.observe()
        if not isinstance(action_index, (int, np.integer)) or not 0 <= action_index < len(menu):
            raise IndexError("action index outside current menu")
        action = menu[action_index][0]
        old_best = self.best_j
        self.commits += 1
        accepted = action.operation == "stop"
        reason = "stop" if accepted else None
        trial_info = None
        if action.operation == "stop":
            self.stop_reason = "policy_stop"
            self.done = self.terminated = True
        else:
            try:
                trial = self.preview(action)
                trial_info = trial.info
                accepted = bool(trial.info["accepted"])
                reason = trial.info["reason"]
                if accepted:
                    self.current_layout = trial.layout
                    self.current_j = trial.score
                    if trial.score < self.best_j - 1e-10:
                        self.best_layout = trial.layout
                        self.best_j = trial.score
            except RuntimeError as exc:
                accepted = False
                reason = str(exc)
        reward = self.config.reward_scale * (old_best - self.best_j)
        if not self.done and (self.commits >= self.config.max_commits or
                              self.trials >= self.config.max_trials or
                              self.geometry_calls + MENU_MATRIX_BOUND + TRIAL_MATRIX_RESERVE > self.config.max_geometry_calls):
            self.stop_reason = ("commit_budget" if self.commits >= self.config.max_commits else
                                "trial_budget" if self.trials >= self.config.max_trials else
                                "geometry_budget")
            self.done = self.truncated = True
        row = self._info("step", request_index=int(action_index), action=action.to_record(),
                         accepted=accepted, reason=reason, reward=reward,
                         trial=trial_info, terminated=self.terminated,
                         truncated=self.truncated,
                         menu_sha256=candidate_sha256(menu))
        self.ledger.append(row)
        next_obs = self.observe()[0] if not self.done else self._failure_observation()
        return next_obs, reward, self.terminated, self.truncated, row

    def result(self):
        status = ("failed_initialization" if self.failure_reason else
                  "completed" if self.done else "incomplete")
        return {**self._info(status),
                "initial_layout": self.initial_layout.to_record() if self.initial_layout else None,
                "raw_terminal_layout": self.current_layout.to_record() if self.current_layout else None,
                "best_layout": self.best_layout.to_record() if self.best_layout else None,
                "initialization": self.initialization,
                "ledger": self.ledger, "terminated": self.terminated,
                "truncated": self.truncated,
                "reward_sum": sum(row["reward"] for row in self.ledger)}


已核冻结源码并装载定义：drl_r04_core 第 51–317 行


`h1_teacher_label` 用实际候选试探选择示范动作；这部分是真实教师代码。I0/H1/H2 是同组对照，不是另三次神经网络训练。


In [7]:
%%project_module satellite_rl.coverage.drl_r04_core 318:389


def run_i0(env: DrlR04Env):
    env.reset()
    if not env.done:
        env.step(0)
    return env.result()


def run_random(env: DrlR04Env, seed: int):
    rng = np.random.default_rng(seed)
    env.reset()
    while not env.done:
        env.step(int(rng.integers(len(env.menu))))
    return env.result()


def h1_teacher_label(env: DrlR04Env):
    """Offline label and charged expert trials; call only outside policy inference."""
    menu = env.menu
    selected, best = 0, env.best_j
    for index, (action, _) in enumerate(menu[1:], 1):
        if env.trials >= env.config.max_trials or env.geometry_calls + TRIAL_MATRIX_RESERVE > env.config.max_geometry_calls:
            break
        trial = env.preview(action)
        if trial.score is not None and trial.score < best - 1e-10:
            selected, best = index, trial.score
    return selected


def run_h1(env: DrlR04Env):
    env.reset()
    while not env.done:
        env.step(h1_teacher_label(env))
    return env.result()


def run_h2(env: DrlR04Env):
    """Two-state beam, depth two; all branches debit the same physical ledger."""
    env.reset()
    while not env.done:
        first_menu = env.menu
        first = []
        for i, (action, _) in enumerate(first_menu[1:], 1):
            if env.trials >= env.config.max_trials or env.geometry_calls + TRIAL_MATRIX_RESERVE > env.config.max_geometry_calls:
                break
            trial = env.preview(action)
            if trial.score is not None:
                first.append((trial.score, i, trial))
        first.sort(key=lambda row: (row[0], row[1]))
        paths = []
        for score, i, trial in first[:2]:
            paths.append((min(env.best_j, score), (i,)))
            if env.config.max_commits - env.commits < 2:
                continue
            branch_menu = env._menu(trial.layout)
            for j, (action, _) in enumerate(branch_menu[1:], 1):
                if env.trials >= env.config.max_trials or env.geometry_calls + TRIAL_MATRIX_RESERVE > env.config.max_geometry_calls:
                    break
                second = env.preview(action, layout=trial.layout)
                if second.score is not None:
                    paths.append((min(env.best_j, score, second.score), (i, j)))
        # STOP is an explicit zero-length path; ties prefer fewer commits.
        best = min(paths + [(env.best_j, ())], key=lambda row: (row[0], len(row[1]), row[1]))
        if not best[1] or best[0] >= env.best_j - 1e-10:
            env.step(0)
            continue
        for index in best[1]:
            if env.done:
                break
            env.step(index)
    return env.result()


已核冻结源码并装载定义：drl_r04_core 第 318–389 行


## 5. 冻结配置、场景流与动作记录

以下开始是工作台的训练入口源码。第一块校验种子 51001、32 路环境、每路 64 步、PPO 学习率 0.0003、4 轮、每批 64 条等参数；正式预算固定为每个 PPO 262144 决策，试点 65536 包含其中。后面的 worker 函数把场景与动作交给上面的环境，在每次真实交互后记录动作菜单、奖励、终态和物理评价，供日志与恢复核验。


In [8]:
%%project_module satellite_rl.coverage.drl_r04_runner 1:149
"""Workbench-only S1-D12A learning, demonstration, and evaluation jobs.

The runner owns persistence and sampling.  Geometry and reward live in
``drl_r04_core``; the network has no dependency on either module.
"""
from __future__ import annotations

from concurrent.futures import ProcessPoolExecutor
from contextlib import ExitStack
from dataclasses import replace
import gzip
import hashlib
import json
import multiprocessing
import os
from pathlib import Path
import sys
import time

import numpy as np
import torch

from .drl_r04_config import DrlR04Config
from .drl_r04_core import (DrlR04Env, h1_teacher_label,
                           run_h1, run_h2, run_i0)
from .drl_r04_policy import DrlR04Policy, Transition, collate, ppo_update
from .runner import (_event, _jsonable, _jsonl, _jsonl_many, _load_torch, _now, _read, _restore_rng,
                     _rng_state, _save_torch, _seed, _sha256, _status,
                     _stop_requested, _write_json)
from .variable_dataset import load_variable_dataset


POLICY_FORMAT = "drl_r04_set_ppo_v1"
CHECKPOINT_FORMAT = "s1_d12a_checkpoint_v1"
MULTIPLIERS = (0.75, 1.0, 1.25)
_RESIDENT_STREAMS = None
_RESIDENT_DATASET = None
_RESIDENT_CORE = None


def _config_hash(config):
    """Canonical schema-6 hash; legacy config_hash validates only older schemas."""
    return hashlib.sha256(json.dumps(config, sort_keys=True, ensure_ascii=False,
                                     separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def _learner_hash(config):
    learning_keys = ("seed", "num_envs", "rollout_steps", "minibatch_size", "epochs",
                     "lr", "gamma", "gae_lambda", "clip_ratio", "entropy_coef",
                     "value_coef", "max_grad_norm")
    return _config_hash({"policy_format": POLICY_FORMAT,
                         "observation_shape": {"beam": 12, "candidate": 16, "global": 14},
                         "network": config["network"],
                         "training": {key: config["training"][key] for key in learning_keys},
                         "core": config["coverage"]["drl_r04"]})


def _write_observation_shard(directory, step, observations):
    """Keep every dynamic network input once, compressed and content-addressed."""
    folder = directory / "observations"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"rollout_{step:06d}.jsonl.gz"
    if path.exists():
        orphan = path.with_name(path.name + ".interrupted." + _sha256(path)[:16])
        if orphan.exists():
            raise FileExistsError("D12-A interrupted observation shard already exists")
        path.replace(orphan)
    temp = path.with_name(path.name + f".tmp.{os.getpid()}")
    try:
        with temp.open("wb") as raw:
            with gzip.GzipFile(filename="", mode="wb", fileobj=raw,
                               compresslevel=3, mtime=0) as packed:
                for item in observations:
                    packed.write((json.dumps(_jsonable(item), ensure_ascii=False,
                                             allow_nan=False, separators=(",", ":")) + "\n").encode())
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)
    return {"file": str(path.relative_to(directory)),
            "sha256": _sha256(path), "rows": len(observations)}


def _dataset(config, role):
    record = config["datasets"][role]
    path = Path(record["path"]).resolve()
    if not path.is_file() or _sha256(path) != record["sha256"]:
        raise ValueError(f"D12-A dataset SHA mismatch: {role}")
    dataset = load_variable_dataset(path)
    if not dataset.scenes:
        raise ValueError(f"D12-A empty dataset: {role}")
    return dataset


def _scaled(scene, multiplier):
    return replace(scene, demands=np.asarray(scene.demands, dtype=np.float64) * multiplier,
                   metadata={**scene.metadata, "demand_multiplier": float(multiplier)})


def _core_config(config):
    values = config.get("coverage", {}).get("drl_r04", {})
    return DrlR04Config(**{key: value for key, value in values.items()
                           if key in DrlR04Config.__dataclass_fields__})


def _reachability_contract(job, config):
    frozen = job.get("reachability_contract")
    if not isinstance(frozen, dict):
        raise ValueError("D12-A formal PPO requires pretraining H1/H2 reachability proof")
    path = Path(frozen.get("path", ""))
    indices = frozen.get("reachable_mother_indices")
    if (not path.is_file() or _sha256(path) != frozen.get("sha256") or
            frozen.get("validation_dataset_sha256") != config["datasets"]["validation"]["sha256"] or
            not isinstance(indices, list) or not indices or
            len(indices) != len(set(indices)) or
            any(type(index) is not int or not 0 <= index < 20 for index in indices)):
        raise ValueError("D12-A frozen reachability proof differs")
    source = _read(path, {})
    if (source.get("reachable_mother_indices") != indices or
            source.get("validation_dataset_sha256") != frozen["validation_dataset_sha256"]):
        raise ValueError("D12-A reachability content differs from frozen job")
    return indices


def _validate_training(config, job, preflight):
    settings = config["training"]
    formal = job.get("experiment_role", "formal") == "formal"
    total = 262144 if formal else 2048
    interval = 16384 if formal else 2048
    expected = {"seed": 51001, "num_envs": 32, "rollout_steps": 64,
                "gamma": 1.0, "gae_lambda": .95, "lr": .0003,
                "epochs": 4, "minibatch_size": 64,
                "entropy_coef": .01, "clip_ratio": .2,
                "value_coef": .5, "max_grad_norm": .5,
                "checkpoint_interval": 2048, "validation_interval": interval,
                "pilot_step": 65536, "cpu_workers": 16}
    for name, value in expected.items():
        if settings.get(name) != value:
            raise ValueError(f"D12-A frozen training {name} differs")
    if settings.get("total_steps") != total:
        raise ValueError("D12-A decision budget differs")
    if settings.get("device") != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("D12-A formal learning requires DQN/CUDA")
    if preflight is None:
        raise RuntimeError("D12-A formal job requires workbench CUDA preflight")
    if settings["total_steps"] % 2048 or settings["validation_interval"] % 2048:
        raise ValueError("D12-A checkpoint boundary differs")
    return settings




已核冻结源码并装载定义：drl_r04_runner 第 1–149 行


In [9]:
%%project_module satellite_rl.coverage.drl_r04_runner 150:236
def _action_digest(env):
    menu = env.menu
    records = [item[0].to_record() for item in menu]
    return hashlib.sha256(json.dumps(records, sort_keys=True,
                                     separators=(",", ":")).encode()).hexdigest()


def _worker_reset(packet):
    scene, core = packet
    env = DrlR04Env(scene, core)
    obs, info = env.reset()
    return env, obs, info


def _worker_step(packet):
    env, choice = packet
    menu_hash = _action_digest(env)
    requested = env.menu[choice][0].to_record()
    before_current, before_best = float(env.current_j), float(env.best_j)
    began = time.monotonic()
    observation, reward, terminated, truncated, info = env.step(choice)
    elapsed = time.monotonic() - began
    physical = _terminal_physics(env) if terminated or truncated else None
    return env, observation, float(reward), bool(terminated), bool(truncated), info, {
        "candidate_menu_sha256": menu_hash, "requested_action": requested,
        "current_j_before": before_current, "best_j_before": before_best,
        "current_j_after": float(env.current_j), "best_j_after": float(env.best_j),
        "execution_seconds": elapsed, "terminal_physics": physical}


def _terminal_physics(env):
    """Measure the raw current and published best terminal layouts exactly once each."""
    from .drl_r04_observation import layout_sha256
    from .variable_downstream import evaluate_layout

    if env.initial_j is None:
        return {"status": "initialization_failed", "current": None, "best": None,
                "physical_allocation_calls": 0}
    layouts = {"initial": env.initial_layout, "current": env.current_layout,
               "best": env.best_layout}
    cache = {}
    aliases = {}
    for name, layout in layouts.items():
        digest = layout_sha256(layout)
        if digest not in cache:
            measured = evaluate_layout(env.scene, layout, {}, route="R04")
            cache[digest] = {"layout_sha256": digest, "beams": measured["beams"],
                             "metrics": measured["metrics"],
                             "timing": measured["timing"], "calls": measured["calls"],
                             "physics_version": measured["physics_version"],
                             "aircraft_service": {key: value for key, value in
                                                  measured["aircraft_service"].items()
                                                  if key != "physical_scene"}}
        aliases[name] = digest
    return {"status": "evaluated_all_actual_beams_and_users",
            "layouts_by_sha256": cache,
            "physical_allocation_calls": len(cache),
            "layout_aliases": aliases}


def _worker_teacher(packet):
    scene, core, remaining_labels, remaining_trials = packet
    env, observation, reset_info = _worker_reset((scene, core))
    labels = []
    began = time.monotonic()
    while not env.done and len(labels) < remaining_labels and env.trials < remaining_trials:
        menu_sha = _action_digest(env)
        before_trials, before_geometry = env.trials, env.geometry_calls
        choice = h1_teacher_label(env)
        previous_observation = observation
        selected_action = env.menu[choice][0].to_record()
        teacher_trials = env.trials - before_trials
        teacher_geometry = env.geometry_calls - before_geometry
        observation, reward, terminated, truncated, step_info = env.step(choice)
        labels.append({"observation": previous_observation, "label": choice,
                      "menu_sha256": menu_sha,
                      "teacher_trials": teacher_trials,
                      "teacher_geometry_calls": teacher_geometry,
                      "selected_action": selected_action,
                      "actual_action": selected_action if step_info["accepted"] else None,
                      "step_info": step_info, "reward": reward,
                      "terminated": terminated, "truncated": truncated})
    return labels, {"result": env.result(), "reset_info": reset_info,
                    "elapsed_seconds": time.monotonic() - began,
                    "complete": bool(env.done)}




已核冻结源码并装载定义：drl_r04_runner 第 150–236 行


`_resident_*` 让每个 CPU 子进程持有固定场景流，训练恢复时仍能还原流的位置。`_fresh_stream` 建立新的训练景；`_terminal` 记录完整回合。每步之后若一景结束，worker 会继续取新景，所以 PPO 的 32 路 rollout 能持续采样。


In [10]:
%%project_module satellite_rl.coverage.drl_r04_runner 237:366
def _resident_init(streams, dataset_record, core):
    """One process owns fixed stream IDs for its entire rollout and resume."""
    global _RESIDENT_STREAMS, _RESIDENT_DATASET, _RESIDENT_CORE
    _RESIDENT_STREAMS = {stream_id: stream for stream_id, stream in streams}
    _RESIDENT_DATASET = _dataset({"datasets": {"train": dataset_record}}, "train")
    _RESIDENT_CORE = core


def _resident_step(packet):
    """Only compact choices/results cross IPC; live environments stay in worker."""
    global_step, choices = packet
    results = []
    for stream_id, choice in choices:
        stream = _RESIDENT_STREAMS[stream_id]
        env, observation, reward, terminated, truncated, info, audit = _worker_step(
            (stream["env"], choice))
        physical = audit.pop("terminal_physics")
        stream["env"], stream["observation"] = env, observation
        stream["episode_return"] += reward
        stream["action_count"] += 1
        scene = _scene_for(stream, _RESIDENT_DATASET)
        event = {"env_step": global_step + stream_id + 1,
                 "stream_id": stream_id,
                 "stream_episode_serial": stream["episode_serial"],
                 **_scene_reference(scene, MULTIPLIERS[stream["level"]]),
                 "candidate_index": int(choice),
                 "actual_action": info.get("action") if info.get("accepted") else None,
                 "reward": reward, "terminated": terminated, "truncated": truncated,
                 "info": info, **audit}
        terminal = None
        if terminated or truncated:
            terminal = _terminal(stream, env_step=global_step + stream_id + 1,
                                 stream_id=stream_id,
                                 reason=info.get("reason", "terminated" if terminated else "budget"),
                                 complete=True)
            terminal["terminal_physics"] = physical
            stream["scene_index"] = int(stream["sampler"].integers(len(_RESIDENT_DATASET.scenes)))
            stream["level"] = int(stream["sampler"].integers(len(MULTIPLIERS)))
            stream["episode_serial"] += 1
            fresh_env, fresh_obs, reset_info = _worker_reset((
                _scene_for(stream, _RESIDENT_DATASET), _RESIDENT_CORE))
            if fresh_env.done:
                raise ValueError(f"D12-A resident initializer failed: {reset_info}")
            stream.update(env=fresh_env, observation=fresh_obs, reset_info=reset_info,
                          episode_return=0.0, action_count=0,
                          episode_start_step=global_step + stream_id + 33)
            observation = fresh_obs
        results.append((stream_id, observation, reward, terminated or truncated,
                        event, terminal))
    return results


def _resident_state(_):
    return list(_RESIDENT_STREAMS.items())


def _resident_tail(last_step):
    rows = []
    for stream_id, stream in _RESIDENT_STREAMS.items():
        if stream["action_count"] == 0:
            env = stream["env"]
            row = env.result()
            row.pop("ledger", None)
            row.update(_scene_reference(env.scene, MULTIPLIERS[stream["level"]]))
            row.update({"stream_id": stream_id,
                        "stream_episode_serial": stream["episode_serial"],
                        "purpose": "budget_tail_snapshot", "episode": None,
                        "completed_episode": False,
                        "stop_reason": "no_action_since_reset",
                        "terminal_env_step": None,
                        "action_trace": {"file": "actions.jsonl", "stream_id": stream_id,
                                         "stream_episode_serial": stream["episode_serial"],
                                         "first_env_step": None, "last_env_step": None,
                                         "count": 0},
                        "terminal_physics": {
                            "status": "not_run_no_action_since_reset",
                            "missing_reason": "no_action_since_reset",
                            "physical_allocation_calls": 0}})
            rows.append(row)
            continue
        last_action_step = last_step - 32 + stream_id + 1
        row = _terminal(stream, env_step=last_action_step, stream_id=stream_id,
                        reason="budget_truncated", complete=False)
        row["terminal_physics"] = _terminal_physics(stream["env"])
        row["episode"] = None
        row["purpose"] = "budget_tail_snapshot"
        rows.append(row)
    return rows


def _scene_for(stream, dataset):
    return _scaled(dataset.scenes[stream["scene_index"]], MULTIPLIERS[stream["level"]])


def _fresh_stream(dataset, core, seed):
    sampler = np.random.default_rng(seed)
    index = int(sampler.integers(len(dataset.scenes)))
    level = int(sampler.integers(len(MULTIPLIERS)))
    scene = _scaled(dataset.scenes[index], MULTIPLIERS[level])
    env, obs, info = _worker_reset((scene, core))
    if env.done:
        raise ValueError(f"D12-A training initializer failed: {scene.scene_id}: {info}")
    return {"sampler": sampler, "scene_index": index, "level": level,
            "env": env, "observation": obs, "episode_serial": 0,
            "episode_return": 0.0, "episode_start_step": 1,
            "action_count": 0, "reset_info": info}


def _scene_reference(scene, multiplier):
    return {"scene_id": scene.scene_id, "scene_sha256": scene.source_hash,
            "demand_multiplier": multiplier}


def _terminal(stream, *, env_step, stream_id, reason, complete):
    env = stream["env"]
    record = env.result()
    # The complete action ledger is in actions.jsonl; keep a precise pointer
    # instead of repeating every step in each terminal row.
    record.pop("ledger", None)
    record.update(_scene_reference(env.scene, MULTIPLIERS[stream["level"]]))
    record.update({"terminal_env_step": env_step, "stream_id": stream_id,
                   "stream_episode_serial": stream["episode_serial"],
                   "episode_return": stream["episode_return"],
                   "stop_reason": reason, "completed_episode": complete,
                   "action_trace": {"file": "actions.jsonl", "stream_id": stream_id,
                                    "stream_episode_serial": stream["episode_serial"],
                                    "first_env_step": stream["episode_start_step"],
                                    "last_env_step": env_step,
                                    "count": stream["action_count"]}})
    return record


已核冻结源码并装载定义：drl_r04_runner 第 237–366 行


## 6. 检查点、模型终点与 P1 的 BC 起点

`_save_checkpoint` 保存策略、优化器、随机状态、32 条环境流和日志计数；`_load_checkpoint` 核任务 ID、配置哈希、源码提交及数据分片，再恢复训练。`_save_model` 写终点权重和来源元数据。`_initialize_p1_from_bc` 严格验证 BC 终点的格式、网络和权重 SHA，随后把 BC 权重载入 P1；P0 不走这一步。


In [11]:
%%project_module satellite_rl.coverage.drl_r04_runner 367:483


def _save_checkpoint(directory, job, config, policy, optimizer, *, streams,
                     env_step, update_step, episode, last_validation, ledgers):
    path = directory / "checkpoint.pt"
    payload = {"format": CHECKPOINT_FORMAT, "run_id": job["run_id"],
               "source_commit": job["git_commit"], "config_hash": _config_hash(config),
               "learner_hyperparameter_hash": _learner_hash(config),
               "policy_format": POLICY_FORMAT, "policy": policy.state_dict(),
               "optimizer": optimizer.state_dict(), "rng": _rng_state(),
               "cpu_workers": config["training"].get("cpu_workers"),
               "cpu_worker_backend": "spawn_resident",
               "streams": streams, "env_step": env_step, "update_step": update_step,
               "episode": episode, "last_validation": last_validation,
               "ledger_counts": dict(ledgers)}
    _save_torch(path, payload)
    return _sha256(path)


def _load_checkpoint(directory, job, config, policy, optimizer):
    payload = _load_torch(directory / "checkpoint.pt", "cpu")
    for key, value in (("format", CHECKPOINT_FORMAT), ("run_id", job["run_id"]),
                       ("source_commit", job["git_commit"]),
                       ("config_hash", _config_hash(config)),
                       ("learner_hyperparameter_hash", _learner_hash(config)),
                       ("policy_format", POLICY_FORMAT),
                       ("cpu_workers", config.get("training", {}).get("cpu_workers"))):
        if payload.get(key) != value:
            raise ValueError(f"D12-A checkpoint {key} differs")
    policy.load_state_dict(payload["policy"], strict=True)
    optimizer.load_state_dict(payload["optimizer"])
    _restore_rng(payload["rng"])
    for name, count in payload["ledger_counts"].items():
        path = directory / name
        rows = path.read_text(encoding="utf-8").splitlines() if path.exists() else []
        if len(rows) < count:
            raise ValueError(f"D12-A ledger shorter than checkpoint: {name}")
        if len(rows) > count:
            path.write_text("\n".join(rows[:count]) + "\n", encoding="utf-8")
    updates_path = directory / "updates.jsonl"
    for line in (updates_path.read_text(encoding="utf-8").splitlines()
                 if updates_path.exists() else []):
        shard = json.loads(line).get("observation_shard", {})
        path = (directory / shard.get("file", "")).resolve()
        if (not path.is_relative_to(directory.resolve()) or not path.is_file() or
                _sha256(path) != shard.get("sha256") or shard.get("rows") != 2048):
            raise ValueError("D12-A committed observation shard differs")
    return payload


def _model_metadata(path, job, config, *, step, selection, preflight):
    return {"policy_format": POLICY_FORMAT, "research_stage": "coverage",
            "run_id": job["run_id"], "route": "D12A", "network": config["network"],
            "train_mode": job.get("train_mode"),
            "bc_source": job.get("bc_source"),
            "training_seed": 51001, "env_step": step, "selection": selection,
            "cpu_workers": config["training"].get("cpu_workers"),
            "cpu_worker_backend": "spawn_resident" if selection != "bc_endpoint" else None,
            "source_commit": job["git_commit"], "config_hash": _config_hash(config),
            "learner_hyperparameter_hash": _learner_hash(config),
            "policy_sha256": _sha256(path), "gpu": (preflight or {}).get("gpu"),
            "torch_version": torch.__version__, "numpy_version": np.__version__,
            "created_at": _now()}


def _save_model(directory, job, config, policy, *, step, selection, preflight):
    folder = {"fixed_budget_endpoint": "final"}.get(selection, selection)
    target = Path(job["model_dir"]) / folder
    target.mkdir(parents=True, exist_ok=True)
    _save_torch(target / "policy.pt", policy.state_dict())
    metadata = _model_metadata(target / "policy.pt", job, config, step=step,
                               selection=selection, preflight=preflight)
    metadata["parameter_count"] = sum(p.numel() for p in policy.parameters())
    _write_json(target / "model.json", metadata)
    _write_json(directory / "artifacts.json", {
        "checkpoint": str(directory / "checkpoint.pt"),
        "final_policy": str(target), "model": str(target / "policy.pt"),
        "model_metadata": str(target / "model.json"),
        "actions": str(directory / "actions.jsonl"),
        "episodes": str(directory / "episodes.jsonl"),
        "budget_tails": str(directory / "budget_tails.jsonl"),
        "updates": str(directory / "updates.jsonl"),
        "evaluations": str(directory / "evaluations.jsonl")})
    return target


def _load_model(job, config, device):
    source = Path(job.get("model_path", job["model_dir"]))
    metadata = _read(source / "model.json", {})
    if (metadata.get("policy_format") != POLICY_FORMAT or
            metadata.get("route") != "D12A" or
            metadata.get("network") != config["network"] or
            metadata.get("learner_hyperparameter_hash") != _learner_hash(config) or
            metadata.get("policy_sha256") != _sha256(source / "policy.pt")):
        raise ValueError("D12-A model provenance differs; old R08 is prohibited")
    policy = DrlR04Policy().to(device)
    policy.load_state_dict(_load_torch(source / "policy.pt", device), strict=True)
    policy.eval()
    return policy, metadata


def _initialize_p1_from_bc(job, config, policy):
    teacher = Path(job["bc_model_dir"])
    metadata = _read(teacher / "model.json", {})
    if (metadata.get("policy_format") != POLICY_FORMAT or
            metadata.get("route") != "D12A" or
            metadata.get("network") != config["network"] or
            metadata.get("learner_hyperparameter_hash") != _learner_hash(config) or
            metadata.get("selection") != "bc_endpoint" or
            metadata.get("rl_env_step") != 0 or
            metadata.get("policy_sha256") != _sha256(teacher / "policy.pt")):
        raise ValueError("P1 requires frozen D12-A BC endpoint")
    policy.load_state_dict(_load_torch(teacher / "policy.pt",
                                       next(policy.parameters()).device), strict=True)
    return metadata




已核冻结源码并装载定义：drl_r04_runner 第 367–483 行


接下来 `_evaluate_policy` 是共用验证/考核函数：按确定性动作选布局并保存逐场景结果。它并不在笔记本执行；定义留在这里，是为了理解训练中验证节点如何调用它。


In [12]:
%%project_module satellite_rl.coverage.drl_r04_runner 484:530
def _evaluate_policy(policy, dataset, core, *, device, step, output,
                     indices=None, baseline=None, completed=0, on_row=None):
    rows = []
    indices = list(range(len(dataset.scenes))) if indices is None else list(indices)
    if len(set(indices)) != len(indices) or any(type(i) is not int or not 0 <= i < len(dataset.scenes)
                                                for i in indices):
        raise ValueError("D12-A scene indices differ")
    ordinal = 0
    for index in indices:
        scene = dataset.scenes[index]
        for multiplier in MULTIPLIERS:
            if ordinal < completed:
                ordinal += 1
                continue
            scaled = _scaled(scene, multiplier)
            env = DrlR04Env(scaled, core)
            began = time.monotonic()
            if baseline is not None:
                {"I0": run_i0, "H1": run_h1, "H2": run_h2}[baseline](env)
                reset_info = None
            else:
                observation, reset_info = env.reset()
                while not env.done:
                    batch = collate([observation], device)
                    with torch.no_grad():
                        choice = int(policy.act(batch, deterministic=True)[0].item())
                    observation, reward, terminated, truncated, info = env.step(choice)
                    if terminated or truncated:
                        break
            algorithm_seconds = time.monotonic() - began
            row = env.result()
            row["terminal_physics"] = _terminal_physics(env)
            row.update(_scene_reference(scaled, multiplier))
            row.update({"dataset_index": index, "checkpoint_step": step,
                        "reset_info": reset_info,
                        "algorithm_seconds": algorithm_seconds,
                        "total_seconds": algorithm_seconds + sum(
                            item["timing"]["total_seconds"] for item in
                            row["terminal_physics"].get("layouts_by_sha256", {}).values())})
            _jsonl(output, row)
            rows.append(row)
            ordinal += 1
            if on_row is not None and not on_row(ordinal):
                return rows
    return rows




已核冻结源码并装载定义：drl_r04_runner 第 484–530 行


## 7. BC：先有教师标签，再拟合教师的动作

`_teacher` 在训练母场景和三档需求上用 H1 真正试探候选，保存最多 20000 条示范、最多 442368 次候选试探的独立账。标签是教师选的当前菜单索引，不是凭空生成的理想束位。`_bc` 逐批读这些标签，使用交叉熵训练同一策略网络，最多 20 轮、1600 次更新，保存 BC 检查点与终点。BC 的 `env_step=0`：它学示范，不把教师试探或 BC 更新冒充 PPO 环境决策。


In [13]:
%%project_module satellite_rl.coverage.drl_r04_runner 531:650
def _teacher(directory, job, config, resume, preflight):
    if preflight is None:
        raise RuntimeError("D12-A teacher requires workbench preflight")
    if job.get("preparation_mode") != "teacher":
        raise ValueError("D12-A prepare mode must be teacher")
    dataset = _dataset(config, "train")
    if len(dataset.scenes) != 576:
        raise ValueError("D12-A teacher requires 576 train mother scenes")
    core = _core_config(config)
    target = directory / "demonstrations"
    target.mkdir(parents=True, exist_ok=True)
    manifest_path = directory / "teacher_manifest.json"
    if manifest_path.exists():
        manifest = _read(manifest_path, {})
        if (not resume or manifest.get("run_id") != job["run_id"] or
                manifest.get("config_hash") != _config_hash(config)):
            raise ValueError("D12-A teacher manifest differs")
    else:
        if resume or any(target.iterdir()):
            raise ValueError("D12-A teacher has incomplete provenance")
        manifest = {"format": "d12a_teacher_v1", "run_id": job["run_id"],
                    "source_commit": job["git_commit"],
                    "config_hash": _config_hash(config),
                    "dataset_sha256": config["datasets"]["train"]["sha256"],
                    "sampling": "all_train_mothers_in_input_order_then_multipliers_0.75_1_1.25",
                    "sampling_seed": None,
                    "max_labels": 20000, "max_trials": 442368,
                    "next_scene": 0, "labels": 0, "trials": 0,
                    "geometry_calls": 0, "unverified_trials": 0,
                    "label_operations": {"stop": 0, "split": 0, "merge": 0},
                    "interrupted_batches": [], "inflight": [], "scenes": []}
        _write_json(manifest_path, manifest)
    if manifest.get("inflight"):
        # A killed process may have spent every reserved trial.  Charge the
        # maximum conservatively; never reuse unknown physical work for free.
        abandoned = manifest.pop("inflight")
        manifest["unverified_trials"] = manifest.get("unverified_trials", 0) + sum(
            item["reserved_trials"] for item in abandoned)
        manifest.setdefault("interrupted_batches", []).append({
            "scenes": abandoned, "reason": "uncommitted_worker_result",
            "recovered_at": _now()})
        _write_json(manifest_path, manifest)
    with ProcessPoolExecutor(max_workers=8,
                             mp_context=multiprocessing.get_context("spawn")) as pool:
        while manifest["next_scene"] < len(dataset.scenes) * 3:
            remaining_labels = 20000 - manifest["labels"]
            remaining_trials = 442368 - manifest["trials"] - manifest.get("unverified_trials", 0)
            if remaining_labels < core.max_commits or remaining_trials < core.max_trials:
                break
            full_parallel = remaining_labels >= 8 * core.max_commits and remaining_trials >= 8 * core.max_trials
            width = 8 if full_parallel else 1
            width = min(width, len(dataset.scenes) * 3 - manifest["next_scene"])
            trial_reservation = core.max_trials
            label_reservation = core.max_commits
            entries, packets = [], []
            for ordinal in range(manifest["next_scene"], manifest["next_scene"] + width):
                scene_index, level = divmod(ordinal, 3)
                scene = _scaled(dataset.scenes[scene_index], MULTIPLIERS[level])
                entries.append({"ordinal": ordinal, "reserved_trials": trial_reservation,
                                "reserved_labels": label_reservation})
                packets.append((scene, core, label_reservation, trial_reservation))
            manifest["inflight"] = entries
            _write_json(manifest_path, manifest)
            for entry, packet, (labels, audit) in zip(entries, packets,
                                                      pool.map(_worker_teacher, packets)):
                ordinal = entry["ordinal"]
                scene_index, level = divmod(ordinal, 3)
                scene = packet[0]
                file = target / f"scene_{ordinal:04d}.pt"
                if file.exists():
                    orphan = file.with_name(file.stem + ".interrupted." +
                                            _sha256(file)[:16] + file.suffix)
                    if orphan.exists():
                        raise FileExistsError("D12-A teacher orphan evidence already exists")
                    file.replace(orphan)
                _save_torch(file, labels)
                result = audit["result"]
                if len(labels) > entry["reserved_labels"] or result["trials"] > entry["reserved_trials"]:
                    raise ValueError("D12-A teacher worker exceeded reserved budget")
                row = {"ordinal": ordinal, "scene_index": scene_index,
                       "level": level, **_scene_reference(scene, MULTIPLIERS[level]),
                       "path": str(file), "sha256": _sha256(file),
                       "labels": len(labels), "teacher_trials": result["trials"],
                       "geometry_calls": result["geometry_calls"],
                       "matrix_calls": result["matrix_calls"],
                       "ray_pair_checks": result["ray_pair_checks"],
                       "complete": audit["complete"],
                       "elapsed_seconds": audit["elapsed_seconds"],
                       "initial_sha256": result["initial_sha256"],
                       "initial_j": result["initial_j"],
                       "best_j": result["best_j"]}
                manifest["scenes"].append(row)
                manifest["labels"] += len(labels)
                for item in labels:
                    operation = item["selected_action"]["operation"]
                    manifest["label_operations"][operation] += 1
                manifest["trials"] += result["trials"]
                manifest["geometry_calls"] += result["geometry_calls"]
                manifest["next_scene"] = ordinal + 1
                manifest["inflight"] = [item for item in manifest["inflight"]
                                        if item["ordinal"] != ordinal]
                _write_json(manifest_path, manifest)
                _status(directory, job, stage="preparing", scene_count=ordinal + 1,
                        labels=manifest["labels"], candidate_trials=manifest["trials"],
                        unverified_candidate_trial_reservation=manifest["unverified_trials"])
            if _stop_requested(directory):
                _status(directory, job, stage="stopped", stop_reason="requested",
                        labels=manifest["labels"], candidate_trials=manifest["trials"],
                        unverified_candidate_trial_reservation=manifest["unverified_trials"])
                return
    manifest["completed"] = True
    _write_json(manifest_path, manifest)
    _write_json(directory / "artifacts.json", {
        "teacher_manifest": str(manifest_path), "demonstrations": str(target)})
    _status(directory, job, stage="completed", labels=manifest["labels"],
            candidate_trials=manifest["trials"],
            unverified_candidate_trial_reservation=manifest["unverified_trials"],
            scene_count=manifest["next_scene"])




已核冻结源码并装载定义：drl_r04_runner 第 531–650 行


In [14]:
%%project_module satellite_rl.coverage.drl_r04_runner 651:749
def _bc(directory, job, config, resume, preflight):
    if preflight is None or config["training"]["device"] != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("D12-A BC requires workbench DQN/CUDA preflight")
    for key, value in (("bc_epochs", 20), ("bc_batch_size", 256),
                       ("bc_max_updates", 1600), ("seed", 51001)):
        if config["training"].get(key) != value:
            raise ValueError(f"D12-A BC frozen {key} differs")
    demonstrations_dir = Path(job["demonstrations_dir"])
    teacher_source = job.get("teacher_source", {})
    manifest_path = Path(teacher_source.get("manifest_path",
                                            demonstrations_dir.parent / "teacher_manifest.json"))
    manifest = _read(manifest_path, {})
    if (manifest.get("format") != "d12a_teacher_v1" or
            not manifest.get("completed") or
            manifest.get("dataset_sha256") != config["datasets"]["train"]["sha256"] or
            manifest.get("labels", 0) > 20000 or
            sum(manifest.get("label_operations", {}).values()) != manifest.get("labels") or
            manifest.get("trials", 0) + manifest.get("unverified_trials", 0) > 442368):
        raise ValueError("D12-A BC requires completed matching teacher artifact")
    demonstrations = []
    for row in manifest["scenes"]:
        file = Path(row["path"])
        if not file.resolve().is_relative_to(demonstrations_dir.resolve()):
            raise ValueError("D12-A teacher shard escaped demonstration directory")
        if _sha256(file) != row["sha256"]:
            raise ValueError("D12-A demonstration shard SHA differs")
        demonstrations.extend(_load_torch(file, "cpu"))
    if len(demonstrations) != manifest["labels"] or not demonstrations:
        raise ValueError("D12-A teacher label count differs")
    _seed(51001)
    policy = DrlR04Policy().to("cuda")
    optimizer = torch.optim.Adam(policy.parameters(), lr=.0003)
    step, epoch = 0, 0
    checkpoint = directory / "bc_checkpoint.pt"
    if resume:
        saved = _load_torch(checkpoint, "cpu")
        if (saved.get("format") != "d12a_bc_v1" or
                saved.get("run_id") != job["run_id"] or
                saved.get("teacher_manifest_sha256") != _sha256(manifest_path)):
            raise ValueError("D12-A BC checkpoint provenance differs")
        policy.load_state_dict(saved["policy"], strict=True)
        optimizer.load_state_dict(saved["optimizer"])
        _restore_rng(saved["rng"])
        step, epoch = saved["update_step"], saved["epoch"]
        updates_path = directory / "bc_updates.jsonl"
        lines = updates_path.read_text(encoding="utf-8").splitlines() if updates_path.exists() else []
        if len(lines) < epoch:
            raise ValueError("D12-A BC update ledger shorter than checkpoint")
        if len(lines) > epoch:
            updates_path.write_text("\n".join(lines[:epoch]) + "\n", encoding="utf-8")
    elif checkpoint.exists() or (directory / "bc_updates.jsonl").exists():
        raise FileExistsError("D12-A BC requires same-ID resume")
    epoch_index = epoch - 1
    for epoch_index in range(epoch, 20):
        permutation = torch.randperm(len(demonstrations)).tolist()
        losses = []
        for offset in range(0, len(permutation), 256):
            if step >= 1600:
                break
            rows = [demonstrations[i] for i in permutation[offset:offset + 256]]
            batch = collate([row["observation"] for row in rows], "cuda")
            labels = torch.as_tensor([row["label"] for row in rows],
                                     dtype=torch.int64, device="cuda")
            logits, _ = policy(batch)
            loss = torch.nn.functional.cross_entropy(logits, labels)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), .5)
            optimizer.step()
            step += 1
            losses.append(float(loss.detach()))
        _jsonl(directory / "bc_updates.jsonl", {"epoch": epoch_index + 1,
                "update_step": step, "labels": len(demonstrations),
                "mean_cross_entropy": float(np.mean(losses)) if losses else None})
        _save_torch(checkpoint, {"format": "d12a_bc_v1", "run_id": job["run_id"],
                    "teacher_manifest_sha256": _sha256(manifest_path),
                    "policy": policy.state_dict(), "optimizer": optimizer.state_dict(),
                    "rng": _rng_state(), "update_step": step, "epoch": epoch_index + 1})
        _status(directory, job, stage="training", update_step=step,
                epoch=epoch_index + 1, label_count=len(demonstrations))
        if _stop_requested(directory):
            _status(directory, job, stage="stopped", stop_reason="requested",
                    update_step=step, epoch=epoch_index + 1)
            return
        if step >= 1600:
            break
    final = _save_model(directory, job, config, policy, step=0,
                        selection="bc_endpoint", preflight=preflight)
    metadata_file = final / "model.json"
    metadata = _read(metadata_file, {})
    metadata.update({"teacher_manifest_sha256": _sha256(manifest_path),
                     "teacher_run_id": manifest["run_id"], "bc_updates": step,
                     "bc_epochs": epoch_index + 1, "bc_labels": len(demonstrations),
                     "rl_env_step": 0})
    _write_json(metadata_file, metadata)
    _status(directory, job, stage="completed", update_step=step,
            epoch=epoch_index + 1, final_policy=str(final))




已核冻结源码并装载定义：drl_r04_runner 第 651–749 行


## 8. 验证节点、科学停止与 PPO 的真实循环

`_validate_if_due` 每 16384 决策检查固定的 20 个验证母场景 × 三档需求；`_scientific_stop_if_due` 负责连续两个验证节点全 STOP 且零进展的停止规则。65536 决策处还要经过既定试点门槛，不能因为看到中途曲线就任意改变预算。


In [15]:
%%project_module satellite_rl.coverage.drl_r04_runner 750:859
def _validate_if_due(directory, job, config, policy, core, dataset, *, step,
                     last_validation, ledgers):
    if step == 0 or step % config["training"]["validation_interval"] or step <= last_validation:
        return last_validation
    # Deterministic argmax evaluation does not consume the training RNG.
    output = directory / "validation" / f"step_{step:06d}.jsonl"
    output.parent.mkdir(parents=True, exist_ok=True)
    if output.exists():
        interrupted = output.with_name(output.stem + ".interrupted." +
                                       _sha256(output)[:16] + output.suffix)
        if interrupted.exists():
            raise FileExistsError("D12-A interrupted validation evidence already exists")
        output.replace(interrupted)
        _event(directory, job, "d12a_validation_restarted_after_uncommitted_node",
               env_step=step, previous_sha256=_sha256(interrupted),
               previous_path=str(interrupted))
    rows = _evaluate_policy(policy, dataset, core, device="cuda", step=step, output=output)
    if len(rows) != 60:
        raise ValueError("D12-A validation node must contain 20 mothers × three levels")
    def refreshed(row):
        return any(item["action"]["operation"] != "stop" and
                   item["best_j"] < row["initial_j"] - 1e-10
                   for item in row.get("ledger", []))

    by_mother = [rows[start:start + 3] for start in range(0, 60, 3)]
    reachable = (_reachability_contract(job, config)
                 if job.get("experiment_role", "formal") == "formal" else [])
    improved_mothers = sum(
        all(row["initial_j"] is not None and row["best_j"] is not None and
            np.isfinite(row["initial_j"]) and np.isfinite(row["best_j"])
            for row in group) and
        float(np.mean([row["initial_j"] - row["best_j"] for row in group])) > 1e-8 and
        any(refreshed(row) for row in group)
        for group in by_mother)
    feasible_rows = sum(
        bool(row.get("initialization", {}).get("metrics", {}).get("coverage_100")) and
        bool(row.get("initialization", {}).get("metrics", {}).get("demand_conserved")) and
        row["best_j"] is not None and np.isfinite(row["best_j"]) and
        row["geometry_calls"] <= core.max_geometry_calls and
        row["trials"] <= core.max_trials and row["commits"] <= core.max_commits
        for row in rows)
    summary = {"env_step": step, "rows": len(rows), "path": str(output),
               "sha256": _sha256(output),
               "mean_best_j": (float(np.mean([row["best_j"] for row in rows
                                               if row["best_j"] is not None]))
                               if any(row["best_j"] is not None for row in rows) else None),
               "first_step_stop_count": sum(bool(row.get("ledger") and
                                            row["ledger"][0]["action"]["operation"] == "stop")
                                            for row in rows),
               "best_j_improved_count": sum(row["best_j"] is not None and
                                            row["best_j"] < row["initial_j"] - 1e-10
                                            for row in rows),
               "three_level_improved_mother_count": int(improved_mothers),
               "reachable_mother_indices": reachable,
               "reachable_mother_count": len(reachable),
               "reachable_subset_first_stop_zero_progress": bool(reachable) and all(
                    row.get("ledger") and row["ledger"][0]["action"]["operation"] == "stop" and
                    row["best_j"] is not None and row["initial_j"] is not None and
                    abs(row["best_j"] - row["initial_j"]) <= 1e-10
                    for index in reachable for row in by_mother[index]),
               "nonstop_best_refresh_rows": sum(refreshed(row) for row in rows),
               "feasible_row_count": feasible_rows,
               "all_60_first_step_stop_zero_progress": all(
                    row.get("ledger") and row["ledger"][0]["action"]["operation"] == "stop" and
                    row["best_j"] is not None and row["initial_j"] is not None and
                    abs(row["best_j"] - row["initial_j"]) <= 1e-10
                    for row in rows)}
    _jsonl(directory / "evaluations.jsonl", summary)
    ledgers["evaluations.jsonl"] += 1
    _status(directory, job, stage="training", env_step=step,
            last_validation_step=step, latest_validation=summary)
    return step


def _scientific_stop_if_due(directory, job, config, policy, *, step, preflight,
                            checkpoint_sha256):
    if job.get("experiment_role", "formal") != "formal":
        return False
    _reachability_contract(job, config)
    path = directory / "evaluations.jsonl"
    if not path.is_file():
        return False
    nodes = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    if len(nodes) < 2 or [row["env_step"] for row in nodes[-2:]] != [
            step - config["training"]["validation_interval"], step]:
        return False
    for row in nodes[-2:]:
        result_path = Path(row["path"])
        if (row.get("rows") != 60 or not result_path.is_file() or
                _sha256(result_path) != row.get("sha256")):
            raise ValueError("D12-A scientific-stop validation evidence differs")
    if not all(row.get("all_60_first_step_stop_zero_progress") and
               row.get("reachable_subset_first_stop_zero_progress") and
               row.get("best_j_improved_count") == 0 and
               row.get("feasible_row_count") == 60 for row in nodes[-2:]):
        return False
    selection = ("pilot_endpoint" if step == config["training"]["pilot_step"]
                 else "scientific_stop_endpoint")
    endpoint = _save_model(directory, job, config, policy, step=step,
                           selection=selection, preflight=preflight)
    _status(directory, job, stage="stopped", env_step=step,
            stop_reason="consecutive_validation_all_stop_zero_progress",
            scientific_stop=True,
            checkpoint_sha256=checkpoint_sha256, final_policy=str(endpoint),
            scientific_stop_validation_nodes=[row["env_step"] for row in nodes[-2:]])
    _event(directory, job, "d12a_scientific_stop", env_step=step,
           endpoint=str(endpoint))
    return True




已核冻结源码并装载定义：drl_r04_runner 第 750–859 行


`_train_resident` 是实际 PPO 主循环：32 个环境各收集 64 步，合成 2048 条真实交互；用下一观察的价值估计作 bootstrap；调用前面原版 `ppo_update` 更新参数；然后保存动作、回合、损失日志与包含环境流的检查点。循环按 16384 决策验证，65536 决策可停在试点闸门，满 262144 决策才写正式终点。这里的源码是定义，不会因 Run All 执行训练。


In [16]:
%%project_module satellite_rl.coverage.drl_r04_runner 860:1012
def _train_resident(directory, job, config, preflight, settings, core,
                    validation_set, policy, optimizer, streams, step,
                    updates, episode, last_validation, ledgers):
    count, workers = 32, settings["cpu_workers"]
    observations = [stream["observation"] for stream in streams]
    with ExitStack() as stack:
        pools = [stack.enter_context(ProcessPoolExecutor(
            max_workers=1, mp_context=multiprocessing.get_context("spawn"),
            initializer=_resident_init,
            initargs=([(i, streams[i]) for i in range(worker, count, workers)],
                      config["datasets"]["train"], core)))
                 for worker in range(workers)]

        def snapshot():
            returned = [pool.submit(_resident_state, None) for pool in pools]
            ordered = [None] * count
            for task in returned:
                for stream_id, stream in task.result():
                    ordered[stream_id] = stream
            if any(stream is None for stream in ordered):
                raise ValueError("D12-A resident checkpoint lacks a stream")
            return ordered

        if step and step % settings["validation_interval"] == 0 and step > last_validation:
            # Resume after a worker/host interruption during an uncommitted
            # validation node, before taking any further training action.
            last_validation = _validate_if_due(directory, job, config, policy, core,
                                               validation_set, step=step,
                                               last_validation=last_validation,
                                               ledgers=ledgers)
            sha = _save_checkpoint(directory, job, config, policy, optimizer,
                             streams=streams, env_step=step, update_step=updates,
                             episode=episode, last_validation=last_validation,
                             ledgers=ledgers)
            if _scientific_stop_if_due(directory, job, config, policy, step=step,
                                       preflight=preflight, checkpoint_sha256=sha):
                return
            if step == settings["pilot_step"] and job.get("experiment_role", "formal") == "formal":
                pilot = _save_model(directory, job, config, policy, step=step,
                                    selection="pilot_endpoint", preflight=preflight)
                _status(directory, job, stage="pilot_pending", env_step=step,
                        stop_reason="pilot_gate", checkpoint_sha256=sha,
                        pilot_policy=str(pilot))
                return

        while step < settings["total_steps"]:
            transitions, action_rows, episode_rows = [], [], []
            began = time.monotonic()
            for _ in range(64):
                with torch.no_grad():
                    choice, logp, value = policy.act(collate(observations, "cuda"))
                indices, logps, values = choice.tolist(), logp.tolist(), value.tolist()
                offset = step + len(transitions)
                tasks = [pool.submit(_resident_step, (offset,
                          [(i, indices[i]) for i in range(worker, count, workers)]))
                         for worker, pool in enumerate(pools)]
                results = sorted((result for task in tasks for result in task.result()),
                                 key=lambda row: row[0])
                if len(results) != count:
                    raise ValueError("D12-A resident workers returned incomplete step")
                next_observations = [None] * count
                for i, next_obs, reward, done, event, terminal in results:
                    transitions.append(Transition(observations[i], indices[i],
                                                  logps[i], values[i], reward, done))
                    event.update(candidate_mask=observations[i]["candidate_mask"].tolist(),
                                 choice_log_prob=logps[i], value=values[i])
                    action_rows.append(event)
                    next_observations[i] = next_obs
                    if terminal is not None:
                        episode += 1
                        terminal["episode"] = episode
                        episode_rows.append(terminal)
                observations = next_observations
            with torch.no_grad():
                bootstrap = policy(collate(observations, "cuda"))[1].cpu().numpy()
            summary = ppo_update(policy, optimizer, transitions, bootstrap, device="cuda",
                                 gamma=1.0, gae_lambda=.95, clip_ratio=.2,
                                 entropy_coef=.01, value_coef=.5, max_grad_norm=.5,
                                 epochs=4, minibatch_size=64, num_envs=count)
            step += len(transitions)
            updates += summary["minibatch_updates"]
            shard = _write_observation_shard(directory, step,
                                             [item.observation for item in transitions])
            for row_index, row in enumerate(action_rows):
                row["observation_ref"] = {**shard, "row": row_index}
            _jsonl_many(directory / "actions.jsonl", action_rows)
            _jsonl_many(directory / "episodes.jsonl", episode_rows)
            _jsonl(directory / "updates.jsonl", {"env_step": step,
                    "update_step": updates, "episode": episode,
                    "rollout_seconds": time.monotonic() - began,
                    "cpu_worker_backend": "spawn_resident", "cpu_workers": workers,
                    "observation_shard": shard,
                    **summary})
            ledgers["actions.jsonl"] += len(action_rows)
            ledgers["episodes.jsonl"] += len(episode_rows)
            ledgers["updates.jsonl"] += 1
            streams = snapshot()
            sha = _save_checkpoint(directory, job, config, policy, optimizer,
                                   streams=streams, env_step=step, update_step=updates,
                                   episode=episode, last_validation=last_validation,
                                   ledgers=ledgers)
            _status(directory, job, stage="training", env_step=step,
                    budget_env_steps=settings["total_steps"], update_step=updates,
                    episode=episode, checkpoint_sha256=sha,
                    cpu_worker_backend="spawn_resident", cpu_workers=workers,
                    effective_steps_per_second=len(transitions) /
                    max(time.monotonic() - began, 1e-9))
            if step % settings["validation_interval"] == 0:
                last_validation = _validate_if_due(directory, job, config, policy, core,
                                                   validation_set, step=step,
                                                   last_validation=last_validation,
                                                   ledgers=ledgers)
                sha = _save_checkpoint(directory, job, config, policy, optimizer,
                                       streams=streams, env_step=step, update_step=updates,
                                       episode=episode, last_validation=last_validation,
                                       ledgers=ledgers)
                if _scientific_stop_if_due(directory, job, config, policy, step=step,
                                           preflight=preflight, checkpoint_sha256=sha):
                    return
            if _stop_requested(directory):
                _status(directory, job, stage="stopped", env_step=step,
                        stop_reason="requested", checkpoint_sha256=sha)
                return
            if step == settings["pilot_step"] and job.get("experiment_role", "formal") == "formal":
                pilot = _save_model(directory, job, config, policy, step=step,
                                    selection="pilot_endpoint", preflight=preflight)
                _status(directory, job, stage="pilot_pending", env_step=step,
                        stop_reason="pilot_gate", checkpoint_sha256=sha,
                        pilot_policy=str(pilot))
                return
        if step == settings["total_steps"] and ledgers.get("budget_tails.jsonl", 0) == 0:
            tasks = [pool.submit(_resident_tail, step) for pool in pools]
            tails = sorted((row for task in tasks for row in task.result()),
                           key=lambda row: row["stream_id"])
            _jsonl_many(directory / "budget_tails.jsonl", tails)
            ledgers["budget_tails.jsonl"] = len(tails)
            streams = snapshot()
            _save_checkpoint(directory, job, config, policy, optimizer,
                             streams=streams, env_step=step, update_step=updates,
                             episode=episode, last_validation=last_validation,
                             ledgers=ledgers)
    if step != settings["total_steps"]:
        raise ValueError("D12-A training ended short of frozen budget")
    selection = ("fixed_budget_endpoint" if job.get("experiment_role", "formal") == "formal"
                 else "verification_endpoint")
    final = _save_model(directory, job, config, policy, step=step,
                        selection=selection, preflight=preflight)
    _status(directory, job, stage="completed", env_step=step,
            budget_env_steps=settings["total_steps"], update_step=updates,
            episode=episode, final_policy=str(final))
    _event(directory, job, "d12a_training_completed", env_step=step)




已核冻结源码并装载定义：drl_r04_runner 第 860–1012 行


`_train` 选择 P0 或 P1 起点，核数据与断点，并调用上面的主循环。正式任务实际由 `run_drl_r04_job` 通过工作台 worker 派发；笔记本只展示入口，**不调用该函数**。`_evaluate` 是终点考核路径，供查明验证与训练的边界。


In [17]:
%%project_module satellite_rl.coverage.drl_r04_runner 1013:1066
def _train(directory, job, config, resume, preflight):
    settings = _validate_training(config, job, preflight)
    core = _core_config(config)
    train_set, validation_set = _dataset(config, "train"), _dataset(config, "validation")
    if len(validation_set.scenes) != 20:
        raise ValueError("D12-A validation must have 20 mother scenes")
    if job.get("experiment_role", "formal") == "formal":
        _reachability_contract(job, config)
    _seed(settings["seed"])
    policy = DrlR04Policy().to("cuda")
    optimizer = torch.optim.Adam(policy.parameters(), lr=settings["lr"])
    ledger_names = ("actions.jsonl", "episodes.jsonl", "updates.jsonl",
                    "evaluations.jsonl", "budget_tails.jsonl")
    ledgers = {name: 0 for name in ledger_names}
    if resume:
        saved = _load_checkpoint(directory, job, config, policy, optimizer)
        streams = saved["streams"]
        step, updates, episode = saved["env_step"], saved["update_step"], saved["episode"]
        last_validation, ledgers = saved["last_validation"], saved["ledger_counts"]
        if job.get("experiment_role", "formal") == "formal" and step >= settings["pilot_step"]:
            release = _read(directory / "pilot_gate.json", {})
            passed = (release.get("passed") is True and
                      release.get("run_id") == job["run_id"] and
                      release.get("env_step") == settings["pilot_step"])
            if step == settings["pilot_step"] and last_validation < step:
                # An interrupted validation may be repaired, but it must stop
                # at the pilot gate before consuming decision 65537.
                passed = False
            elif step == settings["pilot_step"]:
                passed = passed and release.get("checkpoint_sha256") == _sha256(
                    directory / "checkpoint.pt")
            if not passed and not (step == settings["pilot_step"] and last_validation < step):
                raise ValueError("D12-A pilot gate release missing or differs")
    else:
        if (directory / "checkpoint.pt").exists() or any((directory / name).exists()
                                                      for name in ledger_names):
            raise FileExistsError("D12-A run directory requires same-ID resume")
        if job.get("train_mode") == "P1":
            _initialize_p1_from_bc(job, config, policy)
        elif job.get("train_mode") != "P0":
            raise ValueError("D12-A train_mode must be P0 or P1")
        streams = [_fresh_stream(train_set, core, settings["seed"] + 101 + i * 1009)
                   for i in range(32)]
        for i, stream in enumerate(streams):
            stream["episode_start_step"] = i + 1
        step = updates = episode = last_validation = 0
    _status(directory, job, stage="training", env_step=step,
            budget_env_steps=settings["total_steps"], update_step=updates,
            episode=episode, num_envs=32)
    return _train_resident(directory, job, config, preflight, settings, core,
                           validation_set, policy, optimizer, streams, step,
                           updates, episode, last_validation, ledgers)




已核冻结源码并装载定义：drl_r04_runner 第 1013–1066 行


In [18]:
%%project_module satellite_rl.coverage.drl_r04_runner 1067:1144
def _evaluate(directory, job, config, resume, preflight):
    baseline = job.get("variant") if job.get("variant") in ("I0", "H1", "H2") else None
    if baseline is None and (config["training"]["device"] != "cuda" or
                             not torch.cuda.is_available()):
        raise RuntimeError("D12-A policy evaluation requires CUDA")
    if preflight is None:
        raise RuntimeError("D12-A evaluation requires workbench preflight")
    policy, metadata = (None, {"env_step": 0}) if baseline else _load_model(job, config, "cuda")
    role = next(iter(config["datasets"]))
    dataset = _dataset(config, role)
    indices = job.get("scene_indices")
    if role == "test" and (indices != list(range(job.get("shard_id", -1), 100, 4)) or
                           len(indices) != 25):
        raise ValueError("D12-A test shard identity differs")
    output = directory / "results.partial.jsonl"
    completed = 0
    if output.exists():
        if not resume:
            raise FileExistsError("Existing D12-A evaluation requires same-ID resume")
        lines = output.read_text(encoding="utf-8").splitlines()
        completed = len(lines)
        if completed > (len(indices) if indices is not None else len(dataset.scenes)) * 3:
            raise ValueError("D12-A evaluation ledger exceeds expected rows")
        for ordinal, line in enumerate(lines):
            row = json.loads(line)
            expected_index = (indices if indices is not None else range(len(dataset.scenes)))[ordinal // 3]
            if row.get("dataset_index") != expected_index or row.get("demand_multiplier") != MULTIPLIERS[ordinal % 3]:
                raise ValueError("D12-A evaluation row prefix differs")
    expected = (len(indices) if indices is not None else len(dataset.scenes)) * 3
    _status(directory, job, stage="evaluating", scene_count=completed,
            expected_scene_count=expected, results_partial=str(output))
    def progress(count):
        _status(directory, job, stage="evaluating", scene_count=count,
                expected_scene_count=expected, results_partial=str(output))
        return not _stop_requested(directory)

    rows = _evaluate_policy(policy, dataset, _core_config(config), device="cuda",
                            step=metadata["env_step"], output=output,
                            indices=indices, baseline=baseline, completed=completed,
                            on_row=progress)
    if len(rows) + completed < expected and _stop_requested(directory):
        _status(directory, job, stage="stopped", stop_reason="requested",
                scene_count=len(rows) + completed, results_partial=str(output))
        return
    if len(rows) + completed != expected:
        raise ValueError("D12-A evaluation row count differs")
    output.replace(directory / "results.jsonl")
    output = directory / "results.jsonl"
    all_rows = [json.loads(line) for line in output.read_text(encoding="utf-8").splitlines()]
    _write_json(directory / "summary.json", {"scene_count": len(all_rows),
                "result_sha256": _sha256(output), "model": metadata,
                "method": job.get("variant"),
                "mean_best_j": (float(np.mean([x["best_j"] for x in all_rows
                                               if x["best_j"] is not None]))
                                if any(x["best_j"] is not None for x in all_rows)
                                else None)})
    _status(directory, job, stage="completed", scene_count=len(all_rows),
            results=str(output))


def run_drl_r04_job(directory, job, config, *, resume=False, preflight=None):
    """Dispatch a frozen D12-A task through the normal workbench worker."""
    directory = Path(directory)
    if config.get("schema_version") != 6 or config.get("research_stage") != "coverage":
        raise ValueError("D12-A requires coverage schema 6")
    directory.mkdir(parents=True, exist_ok=True)
    _status(directory, job, stage="initializing", preflight=preflight)
    if job["kind"] == "train":
        if job.get("train_mode") == "BC":
            _bc(directory, job, config, resume, preflight)
        else:
            _train(directory, job, config, resume, preflight)
    elif job["kind"] == "evaluate":
        _evaluate(directory, job, config, resume, preflight)
    elif job["kind"] == "prepare":
        _teacher(directory, job, config, resume, preflight)
    else:
        raise ValueError("D12-A job kind must be train, prepare or evaluate")


已核冻结源码并装载定义：drl_r04_runner 第 1067–1144 行


## 9. 只读核对：这份代码对应哪些已完成任务

最后只构造 CPU 上的网络对象、统计参数，并读取原运行清单，不建场景、不推进环境、不写模型、不发送任务。正式记录中 BC 实际完成 1580 次更新；P0/P1 各完成 262144 次决策与 16384 次小批更新。它们在同一初解/菜单/目标和独立考核条件下比较；结果见 [正式报告 09](../实验报告/09_R04引导的DRL覆盖A阶段对照.md)。报告结论是：PPO 确实改变了布局，但 P0/P1 均未越过强基线 H1 的既定门槛，P1 在 BC 后继续 PPO 也未增加总体收益。这里不把不同奖励的累计回报跨组排名，不把单种子结果称为稳定性证明。


In [19]:
import sys
from satellite_rl.coverage.drl_r04_config import DrlR04Config
from satellite_rl.coverage.drl_r04_policy import DrlR04Policy

assert all(getattr(sys.modules[name], "__file__", "").startswith(str(FROZEN_SRC))
           for name in SOURCE_RANGES)
contract = DrlR04Config()
network = DrlR04Policy()  # CPU 构造；不加载训练权重
parameter_count = sum(p.numel() for p in network.parameters())
assert parameter_count == 88450
assert manifest["status"] == "delivered"
assert all(manifest["tasks"][v]["stage"] == "completed" for v in ("BC", "P0", "P1"))
print("实验：", plan["experiment_id"], "| 冻结源码：", SCIENTIFIC_COMMIT)
print("预算：PPO 各", plan["budget"]["ppo_each"], "决策；BC 最多", plan["budget"]["bc_updates_max"], "更新")
print("实际：BC", manifest["tasks"]["BC"]["update_step"], "更新；",
      "P0", manifest["tasks"]["P0"]["env_step"], "决策；",
      "P1", manifest["tasks"]["P1"]["env_step"], "决策")
print("CPU 网络参数量：", parameter_count, "| 场景动作上限：", contract.max_commits)
print("本单元没有调用 run_drl_r04_job、_train、_bc、_teacher 或 _evaluate")


实验： S1-D12A | 冻结源码： dfe7c394267e94bd2084ae72add3aa2c855b578f
预算：PPO 各 262144 决策；BC 最多 1600 更新
实际：BC 1580 更新； P0 262144 决策； P1 262144 决策
CPU 网络参数量： 88450 | 场景动作上限： 32
本单元没有调用 run_drl_r04_job、_train、_bc、_teacher 或 _evaluate
